# HVG-5000 vs all-genes UMAP comparison

Uses default paths from `scripts/preprocessing/layout.py`.

Build the all-genes variant once:

```bash
uv run scripts/preprocessing/run_preprocessing.py \
    --variant all_genes --all-drugs --scgpt-python ...
```

In [ ]:
import sys
from pathlib import Path

import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import scanpy as sc

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT))

from scripts.preprocessing.layout import PipelinePaths

HVG_PATH = PipelinePaths.build(None, "hvg5000").targets_h5ad
ALL_GENES_PATH = PipelinePaths.build(None, "all_genes").targets_h5ad

In [ ]:
missing = [p for p in (HVG_PATH, ALL_GENES_PATH) if not p.exists()]
if missing:
    raise FileNotFoundError(
        "Missing:\n  " + "\n  ".join(str(p) for p in missing)
        + "\nRun preprocessing with --variant all_genes if needed."
    )
print(HVG_PATH)
print(ALL_GENES_PATH)

In [ ]:
adata_hvg = sc.read_h5ad(HVG_PATH)
adata_all = sc.read_h5ad(ALL_GENES_PATH)
shared = adata_hvg.obs_names.intersection(adata_all.obs_names)
adata_hvg = adata_hvg[shared].copy()
adata_all = adata_all[shared].copy()
print(adata_hvg.shape, adata_all.shape)

In [ ]:
for tag, ad_ in (("hvg5000", adata_hvg), ("all_genes", adata_all)):
    if "X_pca" not in ad_.obsm:
        sc.pp.normalize_total(ad_, target_sum=1e4)
        sc.pp.log1p(ad_)
        sc.pp.pca(ad_)
    if "X_scGPT" not in ad_.obsm:
        raise RuntimeError(f"[{tag}] missing X_scGPT — run preprocessing for that variant.")

In [ ]:
for label, source, rep in [
    ("pca_hvg", adata_hvg, "X_pca"),
    ("pca_all", adata_all, "X_pca"),
    ("scgpt_hvg", adata_hvg, "X_scGPT"),
    ("scgpt_all", adata_all, "X_scGPT"),
]:
    sc.pp.neighbors(source, use_rep=rep, n_neighbors=15, key_added=label)
    sc.tl.umap(source, neighbors_key=label)
    adata_hvg.obsm[f"X_umap_{label}"] = source.obsm["X_umap"].copy()

In [ ]:
PANELS = [
    ("PCA · HVG-5000", "X_umap_pca_hvg"),
    ("PCA · all genes", "X_umap_pca_all"),
    ("scGPT · HVG-5000", "X_umap_scgpt_hvg"),
    ("scGPT · all genes", "X_umap_scgpt_all"),
]
fig, axes = plt.subplots(2, 2, figsize=(15, 14))
for ax, (title, basis) in zip(axes.flatten(), PANELS):
    sc.pl.embedding(adata_hvg, basis=basis, color="Cancer_type", show=False, ax=ax,
                    title=title, palette="rainbow", legend_loc="none")
cats = adata_hvg.obs["Cancer_type"].cat.categories
colors = adata_hvg.uns["Cancer_type_colors"]
handles = [mlines.Line2D([], [], marker="o", markerfacecolor=colors[i], markersize=8, label=c)
           for i, c in enumerate(cats)]
fig.legend(handles=handles, loc="lower center", bbox_to_anchor=(0.5, 0), ncol=5, frameon=False)
plt.tight_layout(rect=[0, 0.18, 1, 1])
plt.show()

In [ ]:
mask = adata_hvg.obs.get("train_mask_paclitaxel", np.isfinite(adata_hvg.obs["viability_paclitaxel"])) == True
adata_tested = adata_hvg[mask].copy()
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
for ax, (title, basis) in zip(axes.flatten(), PANELS):
    sc.pl.embedding(adata_tested, basis=basis, color="viability_paclitaxel", show=False,
                    ax=ax, title=title, cmap="viridis", colorbar_loc="right")
plt.tight_layout()
plt.show()